European calls, puts and binaries on a single equity

In [ ]:
import numpy as np

def european_option_single_asset(cate, S0, K, r, sigma, T, n_paths=1000000, seed=42):
    np.random.seed(seed)
    # Simulate standard normals
    Z = np.random.randn(n_paths)
    # Simulate terminal price
    ST = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)
    # Payoff
    if cate == "call":
        payoff = np.maximum(ST - K, 0.)
    elif cate == "put":
        payoff = np.maximum(K - ST, 0.)
    elif cate == "binary":
        payoff = (ST > K).astype(float)
    else:
        raise ValueError("Wrong option type argument !!!")
    # Discount
    discounted = np.exp(-r * T) * payoff
    price = np.mean(discounted)
    std_error = np.std(discounted) / np.sqrt(n_paths)
    return price, std_error

In [ ]:
cate = "call"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
n_paths = 1000000
price, std_error = european_option_single_asset(cate, S0, K, r, sigma, T, n_paths)
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

call option: price = 10.43 standard error = 0.01


In [ ]:
cate = "put"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
n_paths = 1000000
price, std_error = european_option_single_asset(cate, S0, K, r, sigma, T, n_paths)
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

put option: price = 5.59 standard error = 0.01


In [ ]:
cate = "binary"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
n_paths = 1000000
price, std_error = european_option_single_asset(cate, S0, K, r, sigma, T, n_paths)
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

binary option: price = 0.53 standard error = 0.00


European calls, puts and binaries on several underlying lognormal equities

In [ ]:
import numpy as np

def european_option_multi_asset(cate, S0, K, r, sigma, corr, w, T, n_paths=1000000, seed=42):
    np.random.seed(seed)
    S0 = np.array(S0)
    sigma = np.array(sigma)
    w = np.array(w)
    d = len(S0)
    # Covariance matrix
    cov = np.outer(sigma, sigma) * corr * T
    # Cholesky factor
    L = np.linalg.cholesky(cov)
    # Simulate independent normals
    Z = np.random.randn(n_paths, d)
    # Correlate
    ST = np.exp(np.log(S0) + (r - 0.5 * sigma**2) * T + np.dot(Z, L.T))
    # Basket payoff
    if cate == "call":
        payoff = np.maximum(np.dot(ST, w) - K, 0.)
    elif cate == "put":
        payoff = np.maximum(K - np.dot(ST, w), 0.)
    elif cate == "binary":
        payoff = (np.dot(ST, w) > K).astype(float)
    else:
        raise ValueError("Wrong option type argument !!!")
    discounted = np.exp(-r * T) * payoff
    price = np.mean(discounted)
    std_error = np.std(discounted) / np.sqrt(n_paths)
    return price, std_error

In [ ]:
cate = "call"
S0 = [100, 100]          # initial prices
sigma = [0.2, 0.2]      # volatilities
corr = [[1., 0.999],     # perfect correlation matrix
        [0.999, 1.]]
w = [0.5, 0.5]          # equal weights
K = 100                 # strike
r = 0.05                # risk-free rate
T = 1.0                 # maturity
n_paths = 1000000
price, std_error = european_option_multi_asset(cate, S0, K, r, sigma, corr, w, T, n_paths)
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

call option: price = 10.43 standard error = 0.01


In [ ]:
cate = "put"
S0 = [100, 100]          # initial prices
sigma = [0.2, 0.2]      # volatilities
corr = [[1., 0.999],     # perfect correlation matrix
        [0.999, 1.]]
w = [0.5, 0.5]          # equal weights
K = 100                 # strike
r = 0.05                # risk-free rate
T = 1.0                 # maturity
n_paths = 1000000
price, std_error = european_option_multi_asset(cate, S0, K, r, sigma, corr, w, T, n_paths)
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

put option: price = 5.58 standard error = 0.01


In [ ]:
cate = "binary"
S0 = [100, 100]          # initial prices
sigma = [0.2, 0.2]      # volatilities
corr = [[1., 0.999],     # perfect correlation matrix
        [0.999, 1.]]
w = [0.5, 0.5]          # equal weights
K = 100                 # strike
r = 0.05                # risk-free rate
T = 1.0                 # maturity
n_paths = 1000000
price, std_error = european_option_multi_asset(cate, S0, K, r, sigma, corr, w, T, n_paths)
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

binary option: price = 0.53 standard error = 0.00


European calls, puts and binaries on several underlying lognormal equities, using low-discrepancy numbers

In [ ]:
import numpy as np
from scipy.stats import qmc, norm

def european_option_multi_asset_ld(cate, S0, K, r, sigma, corr, w, T, n_paths=1000000, seed=42):
    S0 = np.asarray(S0, dtype=float)
    sigma = np.asarray(sigma, dtype=float)
    w = np.asarray(w, dtype=float)
    d = len(S0)
    # covariance
    cov = np.outer(sigma, sigma) * corr * T
    L = np.linalg.cholesky(cov)
    # Halton sampler
    sampler = qmc.Halton(d=d, scramble=True, seed=seed)
    # Generate low-discrepancy points
    U = sampler.random(n=n_paths)
    # Avoid extremes
    U = np.clip(U, 1e-12, 1 - 1e-12)
    # Transform to normal
    Z = norm.ppf(U)
    # Terminal prices
    ST = np.exp(np.log(S0) + (r - 0.5 * sigma**2) * T + np.dot(Z, L.T))
    # Payoff
    if cate == "call":
        payoff = np.maximum(np.dot(ST, w) - K, 0.0)
    elif cate == "put":
        payoff = np.maximum(K - np.dot(ST, w), 0.0)
    elif cate == "binary":
        payoff = (np.dot(ST, w) > K).astype(float)
    else:
        raise ValueError("Wrong option type argument !!!")
    price = np.exp(-r * T) * np.mean(payoff)
    return price

In [ ]:
cate = "call"
S0 = [100, 100]          # initial prices
sigma = [0.2, 0.2]      # volatilities
corr = [[1., 0.9999],     # perfect correlation matrix
        [0.9999, 1.]]
w = [0.5, 0.5]          # equal weights
K = 100                 # strike
r = 0.05                # risk-free rate
T = 1.0                 # maturity
n_paths = 1000000
price = european_option_multi_asset_ld(cate, S0, K, r, sigma, corr, w, T, n_paths)
print(f"{cate} option: price = {price:.2f}")

call option: price = 10.45


In [ ]:
cate = "put"
S0 = [100, 100]          # initial prices
sigma = [0.2, 0.2]      # volatilities
corr = [[1., 0.9999],     # perfect correlation matrix
        [0.9999, 1.]]
w = [0.5, 0.5]          # equal weights
K = 100                 # strike
r = 0.05                # risk-free rate
T = 1.0                 # maturity
n_paths = 1000000
price = european_option_multi_asset_ld(cate, S0, K, r, sigma, corr, w, T, n_paths)
print(f"{cate} option: price = {price:.2f}")

put option: price = 5.57


In [ ]:
cate = "binary"
S0 = [100, 100]          # initial prices
sigma = [0.2, 0.2]      # volatilities
corr = [[1., 0.9999],     # perfect correlation matrix
        [0.9999, 1.]]
w = [0.5, 0.5]          # equal weights
K = 100                 # strike
r = 0.05                # risk-free rate
T = 1.0                 # maturity
n_paths = 1000000
price = european_option_multi_asset_ld(cate, S0, K, r, sigma, corr, w, T, n_paths)
print(f"{cate} option: price = {price:.2f}")

binary option: price = 0.53


Path-dependent options on a single equity

In [ ]:
import numpy as np

def asian_option_single_asset(cate, S0, K, r, sigma, T, n_paths=1000000, n_steps=250, seed=42):
    np.random.seed(seed)
    dt = T / n_steps
    # Simulate Brownian increments
    Z = np.random.randn(n_paths, n_steps)
    # Build log-price paths
    increments = (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z
    log_paths = np.cumsum(increments, axis=1)
    # Convert to price paths
    S_paths = S0 * np.exp(log_paths)
    # Include S0 if you want average from t=0
    # S_paths = np.concatenate([S0 * np.ones((n_paths, 1)), S_paths], axis=1)
    # Arithmetic average
    A = np.mean(S_paths, axis=1)
    if cate == "call":
        payoff = np.maximum(A - K, 0.)
    elif cate == "put":
        payoff = np.maximum(K - A, 0.)
    elif cate == "binary":
        payoff = (A > K).astype(float)
    else:
        raise ValueError("Wrong option type argument !!!")
    discounted = np.exp(-r * T) * payoff
    price = np.mean(discounted)
    std_error = np.std(discounted) / np.sqrt(n_paths)
    return price, std_error

In [ ]:
cate = "call"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
n_paths = 1000000
n_steps = 250
price, std_error = asian_option_single_asset(cate, S0, K, r, sigma, T, n_paths, n_steps)
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

call option: price = 5.76 standard error = 0.01


In [ ]:
cate = "put"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
n_paths = 1000000
n_steps = 250
price, std_error = asian_option_single_asset(cate, S0, K, r, sigma, T, n_paths, n_steps)
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

put option: price = 3.36 standard error = 0.01


In [ ]:
cate = "binary"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
n_paths = 1000000
n_steps = 250
price, std_error = asian_option_single_asset(cate, S0, K, r, sigma, T, n_paths, n_steps)
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

binary option: price = 0.53 standard error = 0.00


Interest rate derivatives

In [ ]:
import numpy as np

def bond(r_paths, dt, t_index, T_index):
    # Integrate r from t to T
    integral_r = np.sum(r_paths[:, t_index:T_index], axis=1) * dt
    # Discount factor
    discount = 1. * np.exp(-integral_r)
    return discount

def vasicek(r0, kappa, theta, sigma, T, n_steps, n_paths, seed=42):
    np.random.seed(seed)
    dt = T / n_steps
    r_paths = np.zeros((n_paths, n_steps + 1))
    r_paths[:, 0] = r0
    for i in range(n_steps):
        Z = np.random.randn(n_paths)
        r_paths[:, i + 1] = r_paths[:, i] + kappa * (theta - r_paths[:, i]) * dt + sigma * np.sqrt(dt) * Z
    return r_paths

def hjm(f0_curve, T_grid, T_max, n_steps, n_paths, sigma, a, seed=42):
    np.random.seed(seed)
    dt = T_max / n_steps
    n_maturities = len(T_grid)
    # Initialize forward curve paths
    f_paths = np.zeros((n_paths, n_steps + 1, n_maturities))
    f_paths[:, 0, :] = f0_curve
    for i in range(n_steps):
        t = i * dt
        Z = np.random.randn(n_paths)
        for j, T in enumerate(T_grid):
            if (T - t) <= 0:
                continue
            # volatility
            vol = sigma * np.exp(-a * (T - t))
            # drift (no-arbitrage condition)
            drift = (sigma**2 / a) * np.exp(-a * (T - t)) * (1 - np.exp(-a * (T - t)))
            f_paths[:, i + 1, j] = (f_paths[:, i, j] + drift * dt + vol * np.sqrt(dt) * Z)
    # short rate = f(t,t)
    r_paths = np.zeros((n_paths, n_steps + 1))
    for i in range(n_steps+1):
        t = i * dt
        idx = np.argmin(np.abs(T_grid - t))
        r_paths[:, i] = f_paths[:, i, idx]
    return r_paths

def bond_option(
        model=None, cate=None, r0=None, kappa=None, theta=None, sigma=None, f0_curve=None, T_grid=None, a=None, T_bond=None, T_option=None, K=None,
        n_steps=None, n_paths=None, seed=42
        ):
    dt = T_bond / n_steps
    # Simulate rates up to T_bond
    if model == "vasicek":
        r_paths = vasicek(r0, kappa, theta, sigma, T_bond, n_steps, n_paths)
    elif model == "hjm":
        r_paths = hjm(f0_curve, T_grid, T_bond, n_steps, n_paths, sigma, a)
    else:
        raise ValueError("Wrong interest rate model argument !!!")
    # Bond price at T_option
    bond_price = bond(r_paths, dt, int(T_option / dt), int(T_bond / dt))
    if cate == "call":
        payoff = np.maximum(bond_price - K, 0.0)
    elif cate == "put":
        payoff = np.maximum(K - bond_price, 0.)
    elif cate == "binary":
        payoff = (bond_price > K).astype(float)
    else:
        raise ValueError("Wrong option type argument !!!")
    # Discount factor along path
    integral_r = np.sum(r_paths[:, :int(T_option / dt)], axis=1) * dt
    discounted = np.exp(-integral_r) * payoff
    price = np.mean(discounted)
    std_error = np.std(discounted) / np.sqrt(n_paths)
    return price, std_error

In [ ]:
model = "vasicek"
cate = "call"
r0 = 0.03
kappa = 0.5
theta = 0.04
sigma = 0.02
T_bond = 2.
T_option = 1.
K = 0.9
n_steps = int(250 * T_bond)
dt = T_bond / n_steps
n_paths = 1000
# r = vasicek(r0, kappa, theta, sigma, T_bond, n_steps, n_paths)
# b = bond(r, dt, int(T_option / dt), int(T_bond / dt))
price, std_error = bond_option(
    model = model, cate = cate, r0 = r0, kappa = kappa, theta = theta, sigma = sigma, T_bond = T_bond, T_option = T_option, K = K,
    n_steps = n_steps, n_paths = n_paths
    )
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

call option: price = 0.06 standard error = 0.00


In [ ]:
model = "vasicek"
cate = "put"
r0 = 0.03
kappa = 0.5
theta = 0.04
sigma = 0.02
T_bond = 2.
T_option = 1.
K = 0.9
n_steps = int(250 * T_bond)
dt = T_bond / n_steps
n_paths = 1000
# r = vasicek(r0, kappa, theta, sigma, T_bond, n_steps, n_paths)
# b = bond(r, dt, int(T_option / dt), int(T_bond / dt))
price, std_error = bond_option(
    model = model, cate = cate, r0 = r0, kappa = kappa, theta = theta, sigma = sigma, T_bond = T_bond, T_option = T_option, K = K,
    n_steps = n_steps, n_paths = n_paths
    )
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

put option: price = 0.00 standard error = 0.00


In [ ]:
model = "vasicek"
cate = "binary"
r0 = 0.03
kappa = 0.5
theta = 0.04
sigma = 0.02
T_bond = 2.
T_option = 1.
K = 0.9
n_steps = int(250 * T_bond)
dt = T_bond / n_steps
n_paths = 1000
# r = vasicek(r0, kappa, theta, sigma, T_bond, n_steps, n_paths)
# b = bond(r, dt, int(T_option / dt), int(T_bond / dt))
price, std_error = bond_option(
    model = model, cate = cate, r0 = r0, kappa = kappa, theta = theta, sigma = sigma, T_bond = T_bond, T_option = T_option, K = K,
    n_steps = n_steps, n_paths = n_paths
    )
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

binary option: price = 0.97 standard error = 0.00


In [ ]:
model = "hjm"
cate = "call"
sigma = 0.02   # volatility level
a = 0.5        # mean reversion (decay)
T_option = 1.
T_bond = 2.
K = 0.9
n_steps = int(250 * T_bond)
n_paths = 1000
# Maturity grid (0 to 5 years)
T_grid = np.linspace(0.0, 4.0, 101)
# Flat forward curve: 3%
f0_curve = 0.03 * np.ones_like(T_grid)
# r = hjm(f0_curve, T_grid, T_bond, n_steps, n_paths, sigma, a)
# b = bond(r, dt, int(T_option / dt), int(T_bond / dt))
price, std_error = bond_option(
    model = model, cate = cate, sigma = sigma, f0_curve = f0_curve, T_grid = T_grid, a = a, T_bond = T_bond, T_option = T_option, K = K,
    n_steps = n_steps, n_paths = n_paths
    )
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

call option: price = 0.08 standard error = 0.00


In [ ]:
model = "hjm"
cate = "put"
sigma = 0.02   # volatility level
a = 0.5        # mean reversion (decay)
T_option = 1.
T_bond = 2.
K = 0.9
n_steps = int(250 * T_bond)
n_paths = 1000
# Maturity grid (0 to 5 years)
T_grid = np.linspace(0.0, 4.0, 101)
# Flat forward curve: 3%
f0_curve = 0.03 * np.ones_like(T_grid)
# r = hjm(f0_curve, T_grid, T_bond, n_steps, n_paths, sigma, a)
# b = bond(r, dt, int(T_option / dt), int(T_bond / dt))
price, std_error = bond_option(
    model = model, cate = cate, sigma = sigma, f0_curve = f0_curve, T_grid = T_grid, a = a, T_bond = T_bond, T_option = T_option, K = K,
    n_steps = n_steps, n_paths = n_paths
    )
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

put option: price = 0.00 standard error = 0.00


In [ ]:
model = "hjm"
cate = "binary"
sigma = 0.02   # volatility level
a = 0.5        # mean reversion (decay)
T_option = 1.
T_bond = 2.
K = 0.9
n_steps = int(250 * T_bond)
n_paths = 1000
# Maturity grid (0 to 5 years)
T_grid = np.linspace(0.0, 4.0, 101)
# Flat forward curve: 3%
f0_curve = 0.03 * np.ones_like(T_grid)
# r = hjm(f0_curve, T_grid, T_bond, n_steps, n_paths, sigma, a)
# b = bond(r, dt, int(T_option / dt), int(T_bond / dt))
price, std_error = bond_option(
    model = model, cate = cate, sigma = sigma, f0_curve = f0_curve, T_grid = T_grid, a = a, T_bond = T_bond, T_option = T_option, K = K,
    n_steps = n_steps, n_paths = n_paths
    )
print(f"{cate} option: price = {price:.2f} standard error = {std_error:.2f}")

binary option: price = 0.98 standard error = 0.00


European call, put, binary options through binomial tree

In [ ]:
import numpy as np

def european_option_single_asset_binomial(cate, S0, K, r, sigma, T, n_steps):
    dt = T / n_steps
    # Up / down factors
    u = np.exp(sigma * np.sqrt(dt))
    d = 1. / u
    # Risk-neutral probability
    p = (np.exp(r * dt) - d) / (u - d)
    # Terminal stock prices
    ST = np.array([S0 * (u**j) * (d**(n_steps - j)) for j in range(n_steps + 1)])
    # Terminal payoff
    if cate == "call":
        payoff = np.maximum(ST - K, 0.)
    elif cate == "put":
        payoff = np.maximum(K - ST, 0.)
    elif cate == "binary":
        payoff = (ST > K).astype(float)
    else:
        raise ValueError("Wrong option type argument !!!")
    # Backward induction
    discount = np.exp(-r * dt)
    for i in range(n_steps, 0, -1):
        payoff = discount * (p * payoff[1:i + 1] + (1 - p) * payoff[0:i])
    return payoff[0]

In [ ]:
cate = "call"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
n_steps = 250
price = european_option_single_asset_binomial(cate, S0, K, r, sigma, T, n_steps)
print(f"{cate} option: price = {price:.2f}")

call option: price = 10.44


In [ ]:
cate = "put"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
n_steps = 250
price = european_option_single_asset_binomial(cate, S0, K, r, sigma, T, n_steps)
print(f"{cate} option: price = {price:.2f}")

put option: price = 5.57


In [ ]:
cate = "binary"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
n_steps = 250
price = european_option_single_asset_binomial(cate, S0, K, r, sigma, T, n_steps)
print(f"{cate} option: price = {price:.2f}")

binary option: price = 0.51


American call, put, binary options through binomial tree

In [ ]:
import numpy as np

def american_option_single_asset_binomial(cate, S0, K, r, sigma, T, n_steps):
    dt = T / n_steps
    u = np.exp(sigma * np.sqrt(dt))
    d = 1. / u
    p = (np.exp(r * dt) - d) / (u - d)
    discount = np.exp(-r * dt)
    # Terminal stock prices
    ST = np.array([S0 * (u**j) * (d**(n_steps - j)) for j in range(n_steps + 1)])
    # Terminal payoff
    if cate == "call":
        values = np.maximum(ST - K, 0.)
    elif cate == "put":
        values = np.maximum(K - ST, 0.)
    elif cate == "binary":
        values = (ST > K).astype(float)
    else:
        raise ValueError("Wrong option type argument !!!")
    # Backward induction with early exercise
    for i in range(n_steps, 0, -1):
        new_values = []
        for j in range(i):
            S = S0 * (u**j) * (d**(i - 1 - j))
            continuation = discount * (p * values[j + 1] + (1 - p) * values[j])
            if cate == "call":
                exercise = max(S - K, 0.)
            elif cate == "put":
                exercise = max(K - S, 0.)
            elif cate == "binary":
                exercise = (S > K).astype(float)
            else:
                raise ValueError("Wrong option type argument !!!")
            new_values.append(max(continuation, exercise))
        values = np.array(new_values)
    return values[0]

In [ ]:
cate = "call"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
n_steps = 250
price = american_option_single_asset_binomial(cate, S0, K, r, sigma, T, n_steps)
print(f"{cate} option: price = {price:.2f}")

call option: price = 10.44


In [ ]:
cate = "put"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
n_steps = 250
price = american_option_single_asset_binomial(cate, S0, K, r, sigma, T, n_steps)
print(f"{cate} option: price = {price:.2f}")

put option: price = 6.09


In [ ]:
cate = "binary"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
n_steps = 250
price = american_option_single_asset_binomial(cate, S0, K, r, sigma, T, n_steps)
print(f"{cate} option: price = {price:.2f}")

binary option: price = 0.96


European call, put, binary options through explicit finite difference

In [ ]:
import numpy as np

def european_option_explicit_fd(cate, S0, K, T, r, sigma, s_steps, t_steps):
    # Grid setup
    S_max = 3 * K
    dS = S_max / s_steps
    dt = T / t_steps
    # Stability condition (important for explicit scheme)
    if dt > (0.5 * dS**2) / (sigma**2 * S_max**2):
        raise ValueError(f"{dt:.4f} > {(0.5 * dS**2) / (sigma**2 * S_max**2):.4f}: Unstable argument !!!")
    S = np.linspace(0, S_max, s_steps + 1)
    if cate == "call":
        V = np.maximum(S - K, 0.)
    elif cate == "put":
        V = np.maximum(K - S, 0.)
    elif cate == "binary":
        V = (S > K).astype(float)
    else:
        raise ValueError("Wrong option type argument !!!")
    # Time stepping (backward)
    for n in range(t_steps):
        V_old = V.copy()
        for i in range(1, s_steps):
            delta = (V_old[i + 1] - V_old[i-1]) / (2 * dS)
            gamma = (V_old[i + 1] - 2 * V_old[i] + V_old[i - 1]) / (dS**2)
            V[i] = V_old[i] + dt * (0.5 * sigma**2 * S[i]**2 * gamma + r * S[i] * delta - r * V_old[i])
        # Boundary conditions
        if cate == "call":
            V[0] = 0.  # S = 0
            V[s_steps] = S_max - K * np.exp(-r * (n + 1) * dt)  # linear asymptotic
        elif cate == "put":
            V[0] = K * np.exp(-r * (n + 1) * dt)
            V[s_steps] = 0.
        elif cate == "binary":
            V[0] = 0.
            V[s_steps] = 1.
        else:
            raise ValueError("Wrong option type argument !!!")
    # Interpolate to get price at S0
    return np.interp(S0, S, V)

In [ ]:
cate = "call"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 200
t_steps = 4000
price = european_option_explicit_fd(cate, S0, K, T, r, sigma, s_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

call option: price = 10.45


In [ ]:
cate = "put"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 200
t_steps = 4000
price = european_option_explicit_fd(cate, S0, K, T, r, sigma, s_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

put option: price = 5.58


In [ ]:
cate = "binary"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 200
t_steps = 4000
price = european_option_explicit_fd(cate, S0, K, T, r, sigma, s_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

binary option: price = 0.54


American call, put, binary options through explicit finite difference

In [1]:
import numpy as np

def american_option_explicit_fd(cate, S0, K, T, r, sigma, s_steps, t_steps):
    S_max = 3 * K
    dS = S_max / s_steps
    dt = T / t_steps
    # Stability condition (important for explicit scheme)
    if dt > (0.5 * dS**2) / (sigma**2 * S_max**2):
        raise ValueError(f"{dt:.4f} > {(0.5 * dS**2) / (sigma**2 * S_max**2):.4f}: Unstable argument !!!")
    S = np.linspace(0, S_max, s_steps + 1)
    if cate == "call":
        V = np.maximum(S - K, 0.)
    elif cate == "put":
        V = np.maximum(K - S, 0.)
    elif cate == "binary":
        V = (S > K).astype(float)
    else:
        raise ValueError("Wrong option type argument !!!")
    for n in range(t_steps):
        V_old = V.copy()
        for i in range(1, s_steps):
            delta = (V_old[i + 1] - V_old[i - 1]) / (2 * dS)
            gamma = (V_old[i + 1] - 2 * V_old[i] + V_old[i - 1]) / (dS**2)
            continuation = V_old[i] + dt * (0.5 * sigma**2 * S[i]**2 * gamma + r * S[i] * delta - r * V_old[i])
            # 🔑 Early exercise condition
            if cate == "call":
                exercise = np.maximum(S[i] - K, 0.)
            elif cate == "put":
                exercise = np.maximum(K - S[i], 0.)
            elif cate == "binary":
                exercise = (S[i] > K).astype(float)
            else:
                raise ValueError("Wrong option type argument !!!")
            V[i] = max(continuation, exercise)
        # Boundary conditions
        if cate == "call":
            V[0] = 0
            V[s_steps] = S_max - K * np.exp(-r * (n + 1) * dt)
        elif cate == "put":
            V[0] = K * np.exp(-r * (n + 1) * dt)
            V[s_steps] = 0.
        elif cate == "binary":
            V[0] = 0.
            V[s_steps] = 1.
        else:
            raise ValueError("Wrong option type argument !!!")
    return np.interp(S0, S, V)

In [2]:
cate = "call"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 200
t_steps = 4000
price = american_option_explicit_fd(cate, S0, K, T, r, sigma, s_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

call option: price = 10.45


In [3]:
cate = "put"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 200
t_steps = 4000
price = american_option_explicit_fd(cate, S0, K, T, r, sigma, s_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

put option: price = 6.09


In [4]:
cate = "binary"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 200
t_steps = 4000
price = american_option_explicit_fd(cate, S0, K, T, r, sigma, s_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

binary option: price = 0.98


European call, put, binary options through cn finite difference

In [9]:
import numpy as np

def european_option_cn_fd(cate, S0, K, T, r, sigma, s_steps, t_steps, omega=1.2, tol=1e-6, max_iter=10000):
    S_max = 3 * K
    dS = S_max / s_steps
    dt = T / t_steps
    S = np.linspace(0, S_max, s_steps + 1)
    # Payoff
    if cate == "call":
        payoff = np.maximum(S - K, 0.)
    elif cate == "put":
        payoff = np.maximum(K - S, 0.)
    elif cate == "binary":
        payoff = (S > K).astype(float)
    else:
        raise ValueError("Wrong option type argument !!!")
    V = payoff.copy()
    # Coefficients
    i = np.arange(1, s_steps)
    alpha = 0.25 * dt * (sigma**2 * i**2 - r * i)
    beta  = -0.5 * dt * (sigma**2 * i**2 + r)
    gamma = 0.25 * dt * (sigma**2 * i**2 + r * i)
    # Tridiagonal matrices A and B
    A_lower = -alpha
    A_diag = 1 - beta
    A_upper = -gamma
    B_lower = alpha
    B_diag = 1 + beta
    B_upper = gamma
    for n in range(t_steps):
        # RHS: B * V^n
        rhs = np.zeros(s_steps - 1)
        for j in range(s_steps - 1):
            rhs[j] = (B_lower[j] * V[j] + B_diag[j]  * V[j + 1] + B_upper[j] * V[j + 2])
        # Boundary conditions contribution
        if cate == "call":
            rhs[0] -= A_lower[0] * 0.  # V(0)=0
            rhs[-1] -= A_upper[-1] * (S_max - K * np.exp(-r * (n + 1) * dt))
        elif cate == "put":
            rhs[0] -= A_lower[0] * K * np.exp(-r * (n + 1) * dt)
            rhs[-1] -= A_upper[-1] * 0.
        elif cate == "binary":
            rhs[0] -= A_lower[0] * 0.  # V(0)=0
            rhs[-1] -= A_upper[-1] * 1.
        else:
            raise ValueError("Wrong option type argument !!!")
        # Initial guess
        V_new = V[1:s_steps].copy()
        # PSOR iteration
        for k in range(max_iter):
            error = 0.
            for j in range(s_steps - 1):
                if j == 0:
                    y = (rhs[j] - A_upper[j] * V_new[j + 1]) / A_diag[j]
                elif j == s_steps - 2:
                    y = (rhs[j] - A_lower[j] * V_new[j - 1]) / A_diag[j]
                else:
                    y = (rhs[j] - A_lower[j] * V_new[j - 1] - A_upper[j] * V_new[j + 1]) / A_diag[j]
                # Relaxation
                y = omega * y + (1 - omega) * V_new[j]
                error = max(error, abs(y - V_new[j]))
                V_new[j] = y
            if error < tol:
                break
        # Update solution
        V[1:s_steps] = V_new
        # Boundary conditions
        if cate == "call":
            V[0] = 0.
            V[s_steps] = S_max - K * np.exp(-r * (n + 1) * dt)
        elif cate == "put":
            V[0] = K * np.exp(-r * (n + 1) * dt)
            V[s_steps] = 0.
        elif cate == "binary":
            V[0] = 0
            V[s_steps] = 1.
        else:
            raise ValueError("Wrong option type argument !!!")
    return np.interp(S0, S, V)

In [10]:
cate = "call"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 250
t_steps = 250
price = european_option_cn_fd(cate, S0, K, T, r, sigma, s_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

call option: price = 10.45


In [11]:
cate = "put"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 250
t_steps = 250
price = european_option_cn_fd(cate, S0, K, T, r, sigma, s_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

put option: price = 5.58


In [12]:
cate = "binary"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 250
t_steps = 250
price = european_option_cn_fd(cate, S0, K, T, r, sigma, s_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

binary option: price = 0.53


American call, put, binary options through cn finite difference

In [5]:
import numpy as np

def american_option_cn_fd(cate, S0, K, T, r, sigma, s_steps, t_steps, omega=1.2, tol=1e-6, max_iter=10000):
    S_max = 3 * K
    dS = S_max / s_steps
    dt = T / t_steps
    S = np.linspace(0, S_max, s_steps + 1)
    # Payoff
    if cate == "call":
        payoff = np.maximum(S - K, 0.)
    elif cate == "put":
        payoff = np.maximum(K - S, 0.)
    elif cate == "binary":
        payoff = (S > K).astype(float)
    else:
        raise ValueError("Wrong option type argument !!!")
    V = payoff.copy()
    # Coefficients
    i = np.arange(1, s_steps)
    alpha = 0.25 * dt * (sigma**2 * i**2 - r * i)
    beta  = -0.5 * dt * (sigma**2 * i**2 + r)
    gamma = 0.25 * dt * (sigma**2 * i**2 + r * i)
    # Tridiagonal matrices A and B
    A_lower = -alpha
    A_diag = 1 - beta
    A_upper = -gamma
    B_lower = alpha
    B_diag = 1 + beta
    B_upper = gamma
    for n in range(t_steps):
        # RHS: B * V^n
        rhs = np.zeros(s_steps - 1)
        for j in range(s_steps - 1):
            rhs[j] = (B_lower[j] * V[j] + B_diag[j]  * V[j + 1] + B_upper[j] * V[j + 2])
        # Boundary conditions contribution
        if cate == "call":
            rhs[0] -= A_lower[0] * 0.  # V(0)=0
            rhs[-1] -= A_upper[-1] * (S_max - K * np.exp(-r * (n + 1) * dt))
        elif cate == "put":
            rhs[0] -= A_lower[0] * K * np.exp(-r * (n + 1) * dt)
            rhs[-1] -= A_upper[-1] * 0.
        elif cate == "binary":
            rhs[0] -= A_lower[0] * 0.  # V(0)=0
            rhs[-1] -= A_upper[-1] * 1.
        else:
            raise ValueError("Wrong option type argument !!!")
        # Initial guess
        V_new = V[1:s_steps].copy()
        # PSOR iteration
        for k in range(max_iter):
            error = 0.
            for j in range(s_steps - 1):
                if j == 0:
                    y = (rhs[j] - A_upper[j] * V_new[j + 1]) / A_diag[j]
                elif j == s_steps - 2:
                    y = (rhs[j] - A_lower[j] * V_new[j - 1]) / A_diag[j]
                else:
                    y = (rhs[j] - A_lower[j] * V_new[j - 1] - A_upper[j] * V_new[j + 1]) / A_diag[j]
                # Relaxation
                y = omega * y + (1 - omega) * V_new[j]
                # Projection (early exercise)
                if cate == "call":
                    exercise = np.maximum(S[j + 1] - K, 0.)
                elif cate == "put":
                    exercise = np.maximum(K - S[j + 1], 0.)
                elif cate == "binary":
                    exercise = (S[j + 1] > K).astype(float)
                else:
                    raise ValueError("Wrong option type argument !!!")
                y = max(y, exercise)
                error = max(error, abs(y - V_new[j]))
                V_new[j] = y
            if error < tol:
                break
        # Update solution
        V[1:s_steps] = V_new
        # Boundary conditions
        if cate == "call":
            V[0] = 0.
            V[s_steps] = S_max - K * np.exp(-r * (n + 1) * dt)
        elif cate == "put":
            V[0] = K * np.exp(-r * (n + 1) * dt)
            V[s_steps] = 0.
        elif cate == "binary":
            V[0] = 0
            V[s_steps] = 1.
        else:
            raise ValueError("Wrong option type argument !!!")
    return np.interp(S0, S, V)

In [6]:
cate = "call"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 250
t_steps = 250
price = american_option_cn_fd(cate, S0, K, T, r, sigma, s_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

call option: price = 10.45


In [7]:
cate = "put"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 250
t_steps = 250
price = american_option_cn_fd(cate, S0, K, T, r, sigma, s_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

put option: price = 6.09


In [8]:
cate = "binary"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 250
t_steps = 250
price = american_option_cn_fd(cate, S0, K, T, r, sigma, s_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

binary option: price = 0.97


Asian call, put, binary options through explicit finite difference

In [22]:
import numpy as np

def asian_option_explicit_fd(cate, S0, K, T, r, sigma, s_steps, a_steps, t_steps):
    S_max = 3 * K * T
    A_max = 3 * K * T
    dS = S_max / s_steps
    dA = A_max / a_steps
    dt = T / t_steps
    if dt > min(dS**2 / sigma**2 / S_max**2, dA / S_max):
        raise ValueError(f"{dt:.4f} > {min(dS**2 / sigma**2 / S_max**2, dA / S_max):.4f}: Unstable argument !!!")
    S = np.linspace(0, S_max, s_steps + 1)
    A = np.linspace(0, A_max, a_steps + 1)
    # Grid: V[i, j] = value at S_i, A_j
    V = np.zeros((s_steps + 1, a_steps + 1))
    # Terminal condition
    for i in range(s_steps + 1):
        for j in range(a_steps + 1):
            if cate == "call":
                V[i, j] = max(A[j] / T - K, 0.)
            elif cate == "put":
                V[i, j] = max(K - A[j] / T, 0.)
            elif cate == "binary":
                V[i, j] = (A[j] / T > K).astype(float)
            else:
                raise ValueError("Wrong option type argument !!!")
    # Backward time stepping
    for n in range(t_steps):
        V_old = V.copy()
        for i in range(1, s_steps):
            for j in range(1, a_steps):
                # Derivatives in S
                dV_dS = (V_old[i + 1, j] - V_old[i - 1, j]) / (2 * dS)
                d2V_dS2 = (V_old[i + 1, j] - 2 * V_old[i, j] + V_old[i - 1, j]) / (dS**2)
                # Derivative in A (upwind scheme is better, but central for simplicity)
                dV_dA = (V_old[i, j + 1] - V_old[i, j - 1]) / (2 * dA)
                V[i, j] = V_old[i, j] + dt * (0.5 * sigma**2 * S[i]**2 * d2V_dS2 + r * S[i] * dV_dS + S[i] * dV_dA - r * V_old[i, j])
        # Boundary conditions
        if cate == "call":
            # S = 0 → option worthless
            V[0, :] = 0.
            # S = S_max → behaves like linear
            for j in range(a_steps + 1):
                V[s_steps, j] = max(A[j] / T - K, 0.)
            # A = 0 → no averaging yet
            V[:, 0] = 0.
            # A = A_max → large average → deep ITM
            for i in range(s_steps + 1):
                V[i, a_steps] = A_max / T - K
        elif cate == "put":
            # S = 0 → option worthless
            for j in range(a_steps + 1):
                V[0, j] = max(K - A[j] / T, 0.)
            V[s_steps, :] = 0.
            # A = 0 → no averaging yet
            for i in range(s_steps + 1):
                V[i, 0] = K - A_max / T
            # A = A_max → large average → deep ITM
            V[:, a_steps] = 0.
        elif cate == "binary":
            # S = 0 → option worthless
            V[0, :] = 0.
            # S = S_max → behaves like linear
            for j in range(a_steps + 1):
                V[s_steps, j] = (A[j] / T > K).astype(float)
            # A = 0 → no averaging yet
            V[:, 0] = 0.
            # A = A_max → large average → deep ITM
            for i in range(s_steps + 1):
                V[i, a_steps] = 1.
        else:
            raise ValueError("Wrong option type argument !!!")
    # Initial condition: A = 0 at t=0
    return np.interp(S0, S, V[:, 0])

In [23]:
cate = "call"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 100
a_steps = 100
t_steps = 1000
price = asian_option_explicit_fd(cate, S0, K, T, r, sigma, s_steps, a_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

call option: price = 0.00


In [24]:
cate = "put"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 100
a_steps = 100
t_steps = 1000
price = asian_option_explicit_fd(cate, S0, K, T, r, sigma, s_steps, a_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

put option: price = -200.00


In [25]:
cate = "binary"
S0 = 100     # initial stock price
K = 100      # strike
r = 0.05     # risk-free rate
sigma = 0.2  # volatility
T = 1.0      # 1 year
s_steps = 100
a_steps = 100
t_steps = 1000
price = asian_option_explicit_fd(cate, S0, K, T, r, sigma, s_steps, a_steps, t_steps)
print(f"{cate} option: price = {price:.2f}")

binary option: price = 0.00


Caps and floors through explicit finite difference

Convertible bonds with stock and spot rate being stochastic explicit finite difference

Convertible bonds with stock and spot rate being stochastic implicit finite difference